# MassMIND LWIR Semantic Segmentation — Final Deliverable

**Authors**: Lucas Coelho, Lucas Aparicio and Lars Husemann
**Course**: Computer Vision — Assignment 2

This notebook is the final deliverable of the 2026 Computer Vision Course at the University of Porto (FEUP). It proposes a custom CV-AI model to perform semantic segmentation on the Long-Wave Infrared image data set (MassMind) captured in and around Boston Harbour. We benchmark our model discuss design choices and results, as well as future outlooks.


## 1. Introduction

Contrary to RGB image data, infrared sensors deliver data from the
emitted heat of objects, resulting in a 1-channel image. Unlike
visible-light cameras, LWIR remains usable at night, in fog and in
glare — exactly the conditions where Autonomous Surface Vehicles need
scene understanding the most.

The **MassMIND** dataset (Nirgudkar et al., *IJRR* 2023) provides 2,916
paired LWIR frames and pixel-level masks across 7 maritime classes (`sky`,
`water`, `bridge`, `obstacle`, `living_obs`, `background`, `self`). It is
the first sizeable public benchmark of its kind.

### Objectives

The assignment requires us to:

1. Propose a **custom** AI model that performs semantic segmentation from
   the LWIR image.
2. **Train and test** it **with and without** data augmentation.
3. Report **IoU** of training and testing, **Precision**, **Recall**, and
   **model complexity** (parameter count).
4. Compare against **at least one existing model** (we use a U-Net + VGG-16
   baseline in two variants — ImageNet-pretrained and from scratch).
5. Discuss the results in the context of the MassMIND paper.

### What this notebook delivers

* Our custom architecture — **`CustomLWIRUNet`** — a from-scratch,
  deliberately lightweight U-Net (~4.7 M parameters) built for single-channel
  LWIR: depthwise-separable convolutions, GroupNorm, SiLU activations, a
  Transformer-encoder bottleneck, and deep-supervision auxiliary heads.
* Two existing-model baselines: a U-Net with a VGG-16 encoder, in
  **pretrained** (ImageNet) and **from-scratch** variants. Same data, same
  loss, same training scaffolding.
* A **3 × 2 ablation**: each of the three models trained with and without
  geometric augmentation (horizontal flip + ±7° rotation), with the
  augmentation **deterministically seeded by (image index, epoch)** so the
  six runs differ only in the controlled variables and are exactly
  reproducible.
* For every run: training and **held-out test** evaluation reporting per-class
  IoU, Precision, Recall, plus a convergence curve, per-class confusion,
  qualitative predictions, and a Pareto view of accuracy vs parameter count.

To fit six runs inside the Kaggle 9 h session limit, images are downsampled
to **384×480** (56 % of the native pixel count → ~½ the per-epoch compute),
each run trains for **35 epochs**, and validation runs every 5 epochs.


## 2. Environment setup

Installs the non-Colab-default packages and performs every import.

In [ ]:
# --- Step 1: install only the four packages Colab does NOT preinstall ------
import importlib, subprocess, sys

def _ensure(pkg_spec: str, import_name: str | None = None) -> None:
    """Pip-install ``pkg_spec`` only when ``import_name`` cannot be imported.

    Makes the cell idempotent on re-run: already-installed packages are
    skipped silently, so a re-run after a fresh Colab restart only pays the
    download cost for the genuinely-missing ones.
    """
    name = import_name or pkg_spec.split('==')[0].split('>=')[0]
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec])

# Colab preinstalls torch, numpy, pandas, matplotlib, tqdm, opencv-python (cv2).
_ensure('segmentation-models-pytorch', 'segmentation_models_pytorch')  # SMP: ready-made U-Net + VGG16
_ensure('albumentations')                                              # data augmentation framework
_ensure('gdown')                                                       # Google Drive download helper
_ensure('tifffile')                                                    # 16-bit TIFF reader (some MassMIND frames)

# --- Step 2: standard library ----------------------------------------------
import os, json, csv, time, logging, random, math, shutil, zipfile
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Final, Literal, Iterable

# --- Step 3: numerical + deep-learning stack -------------------------------
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.flop_counter import FlopCounterMode  # exact FLOPs; built into torch >= 2.0

# --- Step 4: vision-specific libraries -------------------------------------
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# --- Step 5: shared logger and device autodetect ---------------------------
# INFO-level so the trainer's epoch summaries print to the cell output without
# the noise that DEBUG would bring.
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logger = logging.getLogger('massmind')

# On Colab T4 this resolves to CUDA. On a CPU-only laptop the analysis cells
# still run; training cells fall back to CPU (slow but correct).
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__, '  device:', DEVICE, '  CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Filesystem layout for the final deliverable. SEPARATE runs directory
# (runs_final/) so this notebook's artifacts never collide with the earlier
# augmentation-study (notebook 07) or main benchmark (notebook 06).
if Path('/kaggle/working').is_dir():
    ROOT = Path('/kaggle/working')      # Kaggle — persistent, committed on Save
elif Path('/content').is_dir():
    ROOT = Path('/content')             # Colab — ephemeral runtime root
else:
    ROOT = Path.cwd()                   # local
DATA_ROOT = ROOT / 'data' / 'massmind'
SPLIT_PATH = ROOT / 'data' / 'splits' / 'split.json'
STATS_PATH = ROOT / 'data' / 'splits' / 'stats_downsampled.json'
RUNS_DIR = ROOT / 'runs_final'
ANALYSIS_DIR = RUNS_DIR / 'analysis'
for d in (DATA_ROOT.parent, SPLIT_PATH.parent, RUNS_DIR, ANALYSIS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Downsampling target: 384 x 480 (H x W). Exact 0.8 aspect ratio (= 512/640,
# no distortion), both dims divisible by 32 (VGG-16 pools 5x; custom_lwir
# only needs 16). 56.25% of the native pixel count -> ~56% of per-epoch compute.
DOWNSAMPLE_HW = (384, 480)
_native = 512 * 640
print('Working under:', ROOT)
print(f'Downsample to: {DOWNSAMPLE_HW} (H x W) — '
      f'{DOWNSAMPLE_HW[0] * DOWNSAMPLE_HW[1] / _native * 100:.1f}% of native pixels')


## 3. Dataset download

`gdown` fetches the two MassMIND archives from the authors' Google Drive
(<https://github.com/uml-marine-robotics/MassMIND>), unzips, verifies counts.
Idempotent — re-running short-circuits once the 2,916 images and masks are
present.


In [ ]:
# Google Drive file IDs from the MassMIND upstream repo
# (https://github.com/uml-marine-robotics/MassMIND). These are public.
GDRIVE_IDS = {
    'images.zip': '1T572f0oqy5JmuTvVEwkSUeXLWOSHl4hL',
    'masks.zip':  '1pHp480_Q-s72RoDf1nD7ERzsv9yZTDE1',
}
EXPECTED_COUNT = 2916  # per the paper, exactly this many paired image/mask files

def _count_images(directory: Path) -> int:
    """Recursively count .png/.tif/.tiff files under ``directory``."""
    if not directory.exists():
        return 0
    return sum(1 for p in directory.rglob('*') if p.suffix.lower() in {'.png', '.tif', '.tiff'})

def _flatten_single_top_dir(root: Path) -> None:
    """If extraction produced ``root/<single_dir>/...``, move contents up one level.

    Some Drive archives wrap everything inside a top-level folder; we want
    a flat ``data/massmind/data/*.png`` layout so the dataset class finds
    the files at predictable paths.
    """
    entries = [p for p in root.iterdir() if not p.name.startswith('.')]
    if len(entries) == 1 and entries[0].is_dir():
        inner = entries[0]
        for child in inner.iterdir():
            shutil.move(str(child), str(root / child.name))
        inner.rmdir()

def download_massmind(root: Path = DATA_ROOT) -> None:
    """Download + extract both archives into ``root``. Short-circuits if done."""
    image_dir, mask_dir = root / 'data', root / 'mask'
    # Fast path: nothing to do if both directories already have the expected file count.
    if _count_images(image_dir) >= EXPECTED_COUNT and _count_images(mask_dir) >= EXPECTED_COUNT:
        print(f'Dataset already present under {root} — skipping download.')
        return

    import gdown  # imported lazily so the cell still loads on a non-Colab box

    # Stash archives in a sibling directory; we'll delete them after extraction.
    archive_dir = root / '_archives'
    archive_dir.mkdir(parents=True, exist_ok=True)

    # Step A: download each archive. gdown handles Drive's confirm-token
    # redirect automatically; resume=True picks up partial downloads.
    for archive_name, file_id in GDRIVE_IDS.items():
        dest = archive_dir / archive_name
        if not dest.exists() or dest.stat().st_size == 0:
            print(f'Downloading {archive_name} from Drive ({file_id})...')
            gdown.download(id=file_id, output=str(dest), quiet=False, resume=True)
        else:
            print(f'Archive already on disk: {dest.name}')

    # Step B: extract each archive into its target directory.
    print('Extracting images.zip...')
    image_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_dir / 'images.zip') as zf:
        zf.extractall(image_dir)
    print('Extracting masks.zip...')
    mask_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_dir / 'masks.zip') as zf:
        zf.extractall(mask_dir)

    # Step C: normalise any single-top-level-folder nesting from the zip.
    _flatten_single_top_dir(image_dir)
    _flatten_single_top_dir(mask_dir)

    # Step D: sanity-check the extracted counts and free archive disk space.
    n_img, n_msk = _count_images(image_dir), _count_images(mask_dir)
    print(f'Extracted {n_img} images and {n_msk} masks under {root}.')
    if n_img == 0 or n_msk == 0:
        raise RuntimeError(f'Extraction produced no usable files in {root}.')
    shutil.rmtree(archive_dir, ignore_errors=True)  # archives are ~3 GB; not needed again

download_massmind()


## 4. Methodology

### 4.1 Dataset & downsampling

**MassMIND** (Nirgudkar et al., 2023): 2,916 paired 640×512 LWIR frames,
7 maritime classes. Split **70/20/10** train/val/test (2042/583/291 frames),
stratified by the 26 capture sessions (each lowercase-letter prefix in the
filename) so every fold sees frames from every trajectory.

**Downsampling.** Every image and mask is resized to 384×480 *before*
augmentation. Two correctness details:

* **Image → `cv2.INTER_AREA`** resamples by averaging over the source pixel
  area — that *is* a box-filter anti-aliasing lowpass + decimate in one step.
  Decimating without it would alias high-frequency thermal detail.
* **Mask → `cv2.INTER_NEAREST`** — masks are categorical class IDs; averaging
  class 2 and class 5 into 3.5 is meaningless. Nearest-neighbour keeps every
  pixel a valid class ID.

**Normalisation stats are recomputed on the downsampled images.** `INTER_AREA`
preserves the mean but slightly reduces variance, so we recompute std at the
actual training resolution rather than reuse full-res stats.


In [ ]:
# ===========================================================================
# Constants
# ===========================================================================
NUM_CLASSES: Final[int] = 7
MASK_IGNORE_INDEX: Final[int] = 255
CLASS_NAMES: Final[list[str]] = ['sky', 'water', 'bridge', 'obstacle', 'living_obs', 'background', 'self']


# ===========================================================================
# Splits — 70/20/10 stratified by capture session (resolution-independent,
# identical to notebook 06)
# ===========================================================================
def _session_bucket(filename: str) -> str:
    """Lowercase letter prefix of a MassMIND filename, or 'unknown'."""
    stem = Path(filename).stem
    if len(stem) >= 2 and stem[0].isalpha() and stem[0].islower() and stem[1:].isdigit():
        return stem[0]
    return 'unknown'

def generate_splits(data_root: Path, out_path: Path, seed: int = 42,
                    train_frac: float = 0.70, val_frac: float = 0.20) -> dict:
    """Deterministic 70/20/10 split, stratified per capture session."""
    image_dir, mask_dir = data_root / 'data', data_root / 'mask'
    image_names = {p.name for p in image_dir.iterdir() if p.is_file()}
    mask_names = {p.name for p in mask_dir.iterdir() if p.is_file()}
    paired = sorted(image_names & mask_names)
    if not paired:
        raise RuntimeError(f'No paired files under {data_root}')
    buckets: dict[str, list[str]] = {}
    for name in paired:
        buckets.setdefault(_session_bucket(name), []).append(name)
    rng = random.Random(seed)
    splits = {'train': [], 'val': [], 'test': []}
    for items in buckets.values():
        rng.shuffle(items)
        n = len(items)
        n_tr, n_va = round(n * train_frac), round(n * val_frac)
        splits['train'].extend(items[:n_tr])
        splits['val'].extend(items[n_tr:n_tr + n_va])
        splits['test'].extend(items[n_tr + n_va:])
    for k in splits:
        splits[k].sort()
    payload = {'seed': seed, 'counts': {k: len(v) for k, v in splits.items()},
               'splits': splits}
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(payload, indent=2))
    return payload


# ===========================================================================
# Image loader + downsampled normalisation stats
# ===========================================================================
def _load_image_norm(path: Path) -> np.ndarray:
    """Load an image to a 2-D float32 array in [0, 1] (bit-depth aware)."""
    if path.suffix.lower() in {'.tif', '.tiff'}:
        import tifffile
        arr = tifffile.imread(str(path))
    else:
        arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if arr is None:
            raise IOError(str(path))
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    if arr.dtype == np.uint16:
        return arr.astype(np.float32) / 65535.0
    if arr.dtype == np.uint8:
        return arr.astype(np.float32) / 255.0
    return arr.astype(np.float32)

def compute_stats_downsampled(data_root: Path, split_path: Path,
                              out_path: Path, downsample_hw: tuple[int, int]) -> dict:
    """Training-split mean/std, computed on the DOWNSAMPLED images.

    Streaming Welford-style sums in float64 for numerical stability.
    """
    splits = json.loads(split_path.read_text())['splits']
    image_dir = data_root / 'data'
    H, W = downsample_hw
    total_sum = total_sq = 0.0
    total_count = 0
    for name in tqdm(splits['train'], desc='stats', leave=False):
        arr = _load_image_norm(image_dir / name)
        arr = cv2.resize(arr, (W, H), interpolation=cv2.INTER_AREA).astype(np.float64)
        total_sum += float(arr.sum())
        total_sq  += float((arr * arr).sum())
        total_count += int(arr.size)
    mean = total_sum / total_count
    std = float(np.sqrt(max(total_sq / total_count - mean * mean, 0.0)))
    payload = {'mean': float(mean), 'std': std,
               'downsample_hw': list(downsample_hw),
               'n_images': len(splits['train']), 'n_pixels': total_count}
    out_path.write_text(json.dumps(payload, indent=2))
    return payload


# ===========================================================================
# Run splits + downsampled stats now (cheap)
# ===========================================================================
if not SPLIT_PATH.exists():
    generate_splits(DATA_ROOT, SPLIT_PATH)
print('Splits:', {k: len(v) for k, v in json.loads(SPLIT_PATH.read_text())['splits'].items()})

if not STATS_PATH.exists():
    compute_stats_downsampled(DATA_ROOT, SPLIT_PATH, STATS_PATH, DOWNSAMPLE_HW)
print('Stats (downsampled):', json.loads(STATS_PATH.read_text()))


### 4.2 Reproducible augmentation

Augmentation is **horizontal flip + rotation in [−7°, +7°]** — a similar
geometric set to the MassMIND paper Section 5.1 (which uses fixed rotations
of ±2°, ±5°, ±7° plus horizontal mirror). Two augmentations both we and
the paper explicitly *exclude*:

* **Vertical flip** — the maritime horizon is physically fixed (sky on top,
  water below); flipping it would be wrong.
* **Brightness / contrast jitter** — in LWIR the absolute pixel intensity
  *is* the class signal (a warm body is the only cue for `living_obs`).
  The MassMIND paper reports this hurt their results; we follow.

The paper expanded their 2,916 images offline into a set of 40,096
(×13 augmented variants per image). Because of computational constraints on
our side, we instead apply augmentation **on the fly** inside the training
loop — no extra disk footprint, no preprocessing step.

The augmentation itself uses a **deterministic seed** of *(image index,
epoch)* instead of fresh randomness on every load:

```
seed(i, e) = (i·2654435761 + e·40503 + BASE_SEED) mod 2³²
rng        = random.Random(seed(i, e))
flip       = rng.random() < 0.5
angle      = rng.uniform(-7°, +7°)
```

This gives three properties at once:

* **Reproducible** — the same *(image, epoch)* always yields the same
  transform, on every run and every machine. The integer mix avoids Python's
  hash randomisation, so it is robust to process restarts and DataLoader
  worker counts.
* **Identical across models** — the schedule depends only on *(i, e)*, so all
  three models see the exact same augmented stream — a genuinely fair ablation.
* **Still real augmentation** — because the seed includes the epoch, image
  *i* gets a *different* transform each epoch. A fully static per-image
  transform (same flip and angle every epoch for a given image) would just
  bake a fixed pre-rotated dataset and lose augmentation's regularising
  benefit; our scheme keeps it.

The current epoch reaches the dataset (and its DataLoader workers) through
a shared-memory `multiprocessing.Value`, set once per epoch by the trainer
via `dataset.set_epoch(e)`. Geometric transforms are applied with
`cv2.warpAffine` directly — image bilinear + reflect padding, mask
nearest-neighbour + constant 255 border fill so the loss/metric
`ignore_index=255` skips the rotation border.


In [ ]:
import multiprocessing as mp

# ===========================================================================
# Image / mask loaders
# ===========================================================================
def _load_image(path: Path) -> np.ndarray:
    """LWIR image as float32 [H, W] in [0, 1]."""
    return _load_image_norm(path)

def _load_mask(path: Path) -> np.ndarray:
    """Mask as uint8 [H, W] class IDs (0..6 fit in uint8; cv2 ops need it)."""
    arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if arr is None:
        raise IOError(str(path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr.astype(np.uint8)


# ===========================================================================
# Deterministic geometric augmentation
# ===========================================================================
def _aug_seed(idx: int, epoch: int, base: int) -> int:
    """Fixed 32-bit hash of (sample index, epoch, base seed).

    Uses explicit integer arithmetic rather than Python's ``hash`` so the
    result is identical across processes, machines and runs (``hash``
    randomisation affects str/bytes but the constants below are pure ints).
    """
    return (idx * 2654435761 + epoch * 40503 + base) & 0xFFFFFFFF

def _apply_geometric(image: np.ndarray, mask: np.ndarray,
                     do_flip: bool, angle: float) -> tuple[np.ndarray, np.ndarray]:
    """Apply a horizontal flip and/or rotation to image + mask consistently.

    Image: bilinear interpolation, reflect-101 border. Mask: nearest-neighbour,
    constant 255 border fill (rotation-border pixels become ignore_index).
    """
    H, W = image.shape
    if do_flip:
        image = np.ascontiguousarray(image[:, ::-1])
        mask  = np.ascontiguousarray(mask[:, ::-1])
    if abs(angle) > 1e-3:
        M = cv2.getRotationMatrix2D((W / 2.0, H / 2.0), angle, 1.0)
        image = cv2.warpAffine(image, M, (W, H), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_REFLECT_101)
        mask = cv2.warpAffine(mask, M, (W, H), flags=cv2.INTER_NEAREST,
                              borderMode=cv2.BORDER_CONSTANT,
                              borderValue=MASK_IGNORE_INDEX)
    return image, mask


# ===========================================================================
# Dataset — downsampling + deterministic augmentation
# ===========================================================================
class MassMINDDataset(Dataset):
    """LWIR dataset with downsampling and deterministic, reproducible augmentation.

    Per item:
      1. load image (float [0,1]) + mask (uint8 class IDs)
      2. downsample to ``downsample_hw`` — INTER_AREA / INTER_NEAREST
      3. if ``augment``: deterministic flip + rotation seeded by (idx, epoch)
      4. normalise + to-tensor
    """
    def __init__(self, data_root: Path, filenames: list[str], mean: float, std: float,
                 *, downsample_hw: tuple[int, int], augment: bool,
                 aug_base_seed: int = 20260522) -> None:
        self.image_dir = data_root / 'data'
        self.mask_dir = data_root / 'mask'
        self.filenames = list(filenames)
        self.downsample_hw = downsample_hw
        self.augment = augment
        self.aug_base_seed = aug_base_seed
        # Shared-memory epoch counter. Set by the trainer once per epoch; read
        # by every DataLoader worker (mp.Value survives the fork on Linux/Kaggle).
        self._epoch = mp.Value('i', 0)
        # Deterministic tail: normalise (max_pixel_value=1.0 — image is already
        # in [0,1]) then to-tensor. No randomness here.
        self.norm_tf = A.Compose([
            A.Normalize(mean=(mean,), std=(std,), max_pixel_value=1.0),
            ToTensorV2(),
        ])

    def set_epoch(self, epoch: int) -> None:
        """Called by the trainer at the start of every epoch."""
        self._epoch.value = int(epoch)

    def __len__(self) -> int:
        return len(self.filenames)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        name = self.filenames[idx]
        image = _load_image(self.image_dir / name)   # float32 [H0, W0]
        mask  = _load_mask(self.mask_dir / name)     # uint8   [H0, W0]

        # Downsample. INTER_AREA = anti-aliased box-filter decimation.
        H, W = self.downsample_hw
        image = cv2.resize(image, (W, H), interpolation=cv2.INTER_AREA)
        mask  = cv2.resize(mask,  (W, H), interpolation=cv2.INTER_NEAREST)

        # Deterministic per-(image, epoch) augmentation.
        if self.augment:
            rng = random.Random(_aug_seed(idx, self._epoch.value, self.aug_base_seed))
            do_flip = rng.random() < 0.5
            angle = rng.uniform(-7.0, 7.0)
            image, mask = _apply_geometric(image, mask, do_flip, angle)

        out = self.norm_tf(image=image, mask=mask)
        image_t, mask_t = out['image'], out['mask']
        if mask_t.dtype != torch.long:
            mask_t = mask_t.long()
        if image_t.ndim == 2:
            image_t = image_t.unsqueeze(0)
        return {'image': image_t, 'mask': mask_t, 'filename': name}

def collate_fn(batch: list[dict]) -> dict:
    """Stack tensors; keep filenames as a plain list."""
    return {
        'image':     torch.stack([b['image'] for b in batch], dim=0),
        'mask':      torch.stack([b['mask']  for b in batch], dim=0),
        'filenames': [b['filename'] for b in batch],
    }


### 4.3 Models

Three configurations sit on the same training scaffolding (same loss, same
optimiser, same data loaders, same augmentation schedule) so any difference
in val/test mIoU is attributable to architecture and weight initialisation
alone.

#### (a) `vgg16_pretrained` — the SOTA-style baseline

A U-Net with a VGG-16 encoder, ImageNet-pretrained. To accept single-channel
LWIR we surgically replace the first 3-channel conv with a 1-channel one
whose weight is the **channel-mean** of the pretrained RGB filter — this
preserves activation magnitude (sum-init would inflate; fresh-init would
discard everything pretrained).

#### (b) `vgg16_scratch` — the fair-comparison baseline

Same architecture, but `encoder_weights=None`. The first conv is randomly
initialised at 1-channel directly. This isolates how much of the pretrained
model's headroom comes from ImageNet vs. the architecture itself. This is the
closest analogue to the UNet baseline in the MassMIND paper, which they also
train from scratch (Section 5.3).

#### (c) `custom_lwir` — our contribution

A from-scratch model inspired by U-Net.

* **Depthwise-separable convolutions** everywhere except the stem (the stem
  sees 1 channel where depthwise is degenerate). DSConv decomposes a 3×3
  conv into a 3×3 depthwise + 1×1 pointwise — same receptive field, ~8×
  fewer params/FLOPs per block.
* **GroupNorm** instead of BatchNorm — batch-size-independent, deployment-
  friendly, AMP-stable.
* **SiLU (swish)** activations instead of ReLU — smooth gradient, no
  dead-unit failure mode.
* **Transformer bottleneck** — at stride 16 the feature map is 24×30 (~720
  tokens at 384×480 input), small enough for quadratic self-attention. Global
  receptive field at the deepest level helps separate long-range classes.
* **Deep-supervision aux heads** at decoder mid-levels (weights 0.4 / 0.2)
  during training, removed at inference (zero deployment cost).
* **Stem width 48**, channel multipliers `(1, 2, 4, 8, 8)` → widths
  `(48, 96, 192, 384, 384)`, ~4.7 M params total — about **5× smaller** than
  the VGG-16 baselines.

#### Design choices we made

* **Depthwise-separable convolutions.** The U-Net's parameter count is
  dominated by its 3×3 convolutions. MobileNet showed you can decompose a
  3×3 conv into a 3×3 *depthwise* (one filter per input channel) plus a
  1×1 *pointwise* (channel mixing) and end up with the same receptive field
  at roughly 8× fewer parameters and FLOPs. We applied that idea everywhere
  except the stem, where 1-channel input makes the depthwise step degenerate.
* **GroupNorm instead of BatchNorm.** Our hardware constrained us to batch
  size 4. BatchNorm estimates its per-channel statistics from the batch
  itself, so at batch=4 those estimates are noisy and training degrades.
  GroupNorm normalises within each sample (across a group of channels), so
  it doesn't care about batch size at all — exactly what we needed.
* **SiLU instead of ReLU.** ReLU has zero gradient for any pre-activation
  below 0, so a unit whose input stays negative gets no learning signal and
  effectively dies — the "dead ReLU" problem. SiLU (swish, `x · sigmoid(x)`)
  has a smooth, non-zero gradient everywhere, so this can't happen. It
  costs slightly more compute, but the per-block savings from DSConv pay
  for it many times over.
* **Transformer bottleneck.** Self-attention computes pairwise relationships
  between every pair of tokens, which gives a *global* receptive field in a
  single layer — useful for separating long-range classes like `self`
  (vessel mast at the top of the frame) from `obstacle` (boat hull
  mid-frame). The catch is that attention scales as O(N²) in the token
  count N, so it can only be applied where N is small. The deepest stage of
  the U-Net (stride 16, 24×30 = 720 tokens at 384×480) is exactly where N
  is small enough to make it cheap; higher up the U-Net we stick with
  depthwise-separable convolutions.
* **Stem width 48.** The original VGG-16 U-Net was designed for 3-channel
  RGB input and starts at 64 channels. LWIR carries only 1 channel —
  roughly a third of the raw information per pixel — so a narrower model
  is justified. The `stem_channels=48` parameter rescales the *whole* width
  schedule (48 → 96 → 192 → 384 → 384 with our `(1, 2, 4, 8, 8)` multipliers),
  giving us ~4.7 M parameters total versus the VGG-16 U-Net's ~23.75 M.


In [ ]:
# ===========================================================================
# Building blocks for CustomLWIRUNet
# ===========================================================================
#
# All three blocks below pair every Conv2d with a GroupNorm + SiLU, which
# differs from the classic Conv-BN-ReLU recipe in two ways:
#
#   * GroupNorm (instead of BatchNorm):  statistics are per-group not per-
#     batch, so it works at any batch size including 1. Useful for edge
#     deployment and for evaluation runs that aren't bound to the training
#     batch shape.
#   * SiLU / swish (instead of ReLU):    smoother gradient near zero, no
#     dead-unit failure mode. Consistently a small win over ReLU in modern
#     encoder/decoder vision models.
# ===========================================================================

def _safe_num_groups(groups_norm: int, num_channels: int) -> int:
    """Pick a GroupNorm group count that divides ``num_channels`` cleanly.

    nn.GroupNorm requires ``num_channels % num_groups == 0``. We aim for
    ``groups_norm`` (default 8) but back off so each group still has at
    least 4 channels of statistical support — otherwise tiny groups end
    up noisy. If that doesn't divide cleanly, walk down to the largest
    divisor that does.
    """
    target = min(groups_norm, max(num_channels // 4, 1))
    if num_channels % target == 0:
        return target
    for g in range(target, 0, -1):
        if num_channels % g == 0:
            return g
    return 1  # unreachable for num_channels >= 1

def _gn(num_channels: int, groups_norm: int = 8) -> nn.GroupNorm:
    """One-liner for a GroupNorm layer with the channel-safe group count."""
    return nn.GroupNorm(num_groups=_safe_num_groups(groups_norm, num_channels),
                        num_channels=num_channels)


class DepthwiseSeparableConv(nn.Module):
    """3×3 depthwise + 1×1 pointwise, each followed by GroupNorm + SiLU.

    This is the MobileNet trick: decompose a regular 3×3 conv into two cheaper
    stages that together preserve the receptive field but cut parameter and
    FLOP cost by roughly a factor of 9 (≈ ``out_channels / kernel_area``).

        Step 1 (depthwise): one 3×3 filter per input channel — spatial mixing
        Step 2 (pointwise): one 1×1 filter across channels — channel mixing

    Each step gets its own GN + SiLU so the network can normalise/non-linearise
    between them, which is what makes the decomposition work as well as a
    plain conv at much lower cost.
    """
    def __init__(self, in_channels: int, out_channels: int, groups_norm: int = 8) -> None:
        super().__init__()
        # Depthwise: groups=in_channels means each input channel is convolved
        # by exactly one 3×3 filter (independent of the others).
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, padding=1, groups=in_channels, bias=False)
        self.dw_norm = _gn(in_channels, groups_norm)
        self.dw_act = nn.SiLU(inplace=True)
        # Pointwise: a 1×1 conv that mixes information across all channels
        # at every spatial location. This is where the channel-count change
        # (in_channels -> out_channels) actually happens.
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.pw_norm = _gn(out_channels, groups_norm)
        self.pw_act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dw_act(self.dw_norm(self.depthwise(x)))
        return self.pw_act(self.pw_norm(self.pointwise(x)))


class StandardConvBlock(nn.Module):
    """Plain 3×3 Conv + GroupNorm + SiLU. Used only for the stem.

    Why not DepthwiseSeparableConv at the stem? The stem sees a single-channel
    input, so a depthwise conv with groups=1 collapses to a single 3×3 kernel
    per output — which is *exactly* what a regular conv would compute, plus
    an unnecessary pointwise step. Using a plain 3×3 conv here is both
    cleaner and slightly cheaper.
    """
    def __init__(self, in_channels: int, out_channels: int, groups_norm: int = 8) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.norm = _gn(out_channels, groups_norm)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.norm(self.conv(x)))


class DoubleDSConv(nn.Module):
    """Two DepthwiseSeparableConv blocks back-to-back.

    Drop-in replacement for the classic U-Net "DoubleConv" pattern, with the
    convention that channel-count expansion happens in the *first* block (so
    the second block preserves channels). This is the building block of
    every encoder Down stage and every decoder Up stage in CustomLWIRUNet.
    """
    def __init__(self, in_channels: int, out_channels: int, groups_norm: int = 8) -> None:
        super().__init__()
        self.block1 = DepthwiseSeparableConv(in_channels, out_channels, groups_norm)
        self.block2 = DepthwiseSeparableConv(out_channels, out_channels, groups_norm)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block2(self.block1(x))


def init_silu_weights(module: nn.Module) -> None:
    """Kaiming init for all Conv / ConvTranspose layers in ``module``.

    SiLU sits between linear and ReLU near the origin, so PyTorch doesn't ship
    a dedicated nonlinearity name for it. We approximate with leaky_relu with
    a tiny ``a=0.01`` leak — the gain ends up just below the pure-ReLU value,
    which is the right neighborhood for SiLU.

    GroupNorm parameters keep their PyTorch defaults (scale=1, shift=0).
    """
    for m in module.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu', a=0.01)
            if m.bias is not None:
                nn.init.zeros_(m.bias)


In [ ]:
class TransformerBottleneck(nn.Module):
    """A small ``nn.TransformerEncoder`` stack operating on the bottleneck.

    Why a transformer here, and only here? Self-attention is quadratic in the
    number of tokens. At our deepest stage the feature map is 32×40 = 1,280
    tokens (stride 16 from 512×640 input) — small enough to be cheap. Higher
    up the U-Net the spatial grids are 4×, 16×, 64× bigger, where attention
    would be prohibitive.

    What it buys us: a *global* receptive field at the deepest level. Pure
    convolutional U-Nets max out at receptive fields determined by the depth
    and dilation schedule; attention sees every other token in one hop. This
    helps separate long-range classes like "self" (vessel mast at top of
    frame) from "obstacle" (boat hull in the middle) when they share
    similar local thermal signatures.

    Shape contract: (B, C, H, W) in -> (B, C, H, W) out. Channels are
    preserved; this module is a drop-in replacement for the conv body of a
    U-Net bottleneck.
    """
    def __init__(self, channels: int, num_heads: int = 8, num_layers: int = 2,
                 mlp_ratio: int = 2, spatial_size: int = 16) -> None:
        super().__init__()
        if channels % num_heads != 0:
            raise ValueError(f'channels {channels} not divisible by num_heads {num_heads}')
        self.channels = channels
        self.spatial_size = spatial_size

        # 2D learnable positional embedding, stored as [1, C, H, W] for easy
        # broadcasting + bilinear interpolation if the runtime spatial size
        # differs from spatial_size (e.g. non-default input resolution).
        self.pos_embed = nn.Parameter(torch.zeros(1, channels, spatial_size, spatial_size))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # Pre-norm (norm_first=True) is the modern recipe — more stable than
        # post-norm at training-from-scratch. GELU is the standard transformer
        # activation. Dropout 0 because we get plenty of regularisation from
        # the augmentation pipeline + the depthwise convs themselves.
        layer = nn.TransformerEncoderLayer(
            d_model=channels, nhead=num_heads, dim_feedforward=channels * mlp_ratio,
            activation='gelu', batch_first=True, norm_first=True, dropout=0.0,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        # Step 1: positional embedding. If the input spatial size matches the
        # stored embedding, just use it; otherwise bilinearly interpolate.
        if (h, w) == (self.spatial_size, self.spatial_size):
            pos = self.pos_embed
        else:
            pos = F.interpolate(self.pos_embed, size=(h, w), mode='bilinear', align_corners=False)
        x = x + pos
        # Step 2: reshape (B, C, H, W) -> (B, H*W, C) so attention sees tokens.
        tokens = x.flatten(2).transpose(1, 2)
        # Step 3: stacked transformer encoder layers.
        tokens = self.encoder(tokens)
        # Step 4: reshape back to a spatial feature map for the decoder.
        return tokens.transpose(1, 2).reshape(b, c, h, w)


In [ ]:
def adapt_conv_to_one_channel(conv: nn.Conv2d) -> nn.Conv2d:
    """Channel-mean-init a 1-channel Conv2d from an ImageNet-pretrained 3-channel one.

    The intuition: a pretrained RGB filter ``W[out, 3, k, k]`` has learned to
    look at structure that the *sum* of R, G, B exposes. For a grayscale /
    LWIR input there's only one channel, so we replace the [out, 3, k, k]
    weight with the *mean* across the input-channel axis: [out, 1, k, k].

    Why mean and not sum: summing inflates the activation magnitude by 3× and
    breaks the rest of the network (which was tuned to a specific output
    scale). Mean preserves activation magnitude exactly when the new
    grayscale input has the same per-pixel intensity as ``(R+G+B)/3`` — which
    is exactly how grayscale is conventionally defined.

    Random re-initialisation is the alternative, but it discards everything
    pretrained at layer 0, killing roughly half the early-feature transfer.
    """
    if conv.in_channels != 3 or conv.groups != 1:
        raise ValueError('expected 3-channel ungrouped Conv2d')
    new_conv = nn.Conv2d(1, conv.out_channels, conv.kernel_size, stride=conv.stride,
                         padding=conv.padding, dilation=conv.dilation, bias=conv.bias is not None)
    # Copy the channel-averaged weight into the new (untrained) Conv2d.
    with torch.no_grad():
        new_conv.weight.copy_(conv.weight.mean(dim=1, keepdim=True))
        if conv.bias is not None:
            new_conv.bias.copy_(conv.bias)
    return new_conv

def _replace_first_vgg16_conv(model: nn.Module, new_conv: nn.Conv2d) -> None:
    """Surgically swap the first Conv2d of SMP's VGG-16 encoder.

    SMP exposes the VGG-16 encoder as ``model.encoder.features``, a
    ``nn.Sequential`` whose index ``[0]`` is the first conv layer (the one
    that originally sees 3-channel RGB input).
    """
    model.encoder.features[0] = new_conv


In [ ]:
def build_unet_vgg16(num_classes: int = NUM_CLASSES, in_channels: int = 1,
                    encoder_weights: str | None = 'imagenet') -> nn.Module:
    """Construct a U-Net + VGG-16 encoder, first conv adapted to 1-channel input.

    Why we build with ``in_channels=3`` and then swap layer 0: SMP runs its
    own first-layer adaptation logic when you pass ``in_channels != 3``, but
    the strategy isn't guaranteed to be channel-mean (it can be sum-based or
    partly random depending on the version). Building with 3-channel first
    lets the ImageNet weights load cleanly, then we apply our explicit and
    auditable channel-mean adaptation.
    """
    model = smp.Unet(
        encoder_name='vgg16',
        encoder_weights=encoder_weights,
        in_channels=3,             # build with RGB so the pretrained weights load cleanly
        classes=num_classes,
    )
    if in_channels == 1:
        first = model.encoder.features[0]
        if encoder_weights is None:
            # Random init: channel-mean adaptation is meaningless (mean of noise
            # is noise). Replace with a fresh 1-channel Conv2d so random init
            # actually runs on the right kernel shape.
            new_conv = nn.Conv2d(1, first.out_channels, first.kernel_size,
                                 stride=first.stride, padding=first.padding,
                                 bias=first.bias is not None)
        else:
            new_conv = adapt_conv_to_one_channel(first)
        _replace_first_vgg16_conv(model, new_conv)
    return model


In [ ]:
# ===========================================================================
# CustomLWIRUNet — the contribution
# ===========================================================================
#
# Architecture summary (stem 48, multipliers (1,2,4,8,8) -> widths (48,96,192,384,384)):
#
#   [B, 1, H, W]
#   |--- Stem (full res)           : 2× StandardConvBlock(1->48)
#   |--- Encoder
#   |    MaxPool/2 + DoubleDSConv  : 48 -> 96   (stride 2)   skip[1]
#   |    MaxPool/2 + DoubleDSConv  : 96 -> 192  (stride 4)   skip[2]
#   |    MaxPool/2 + DoubleDSConv  : 192 -> 384 (stride 8)   skip[3]
#   |    MaxPool/2 + DoubleDSConv  : 384 -> 384 (stride 16)  -> bottleneck input
#   |--- Bottleneck (no pool)
#   |    TransformerBottleneck     : 384ch, 2 layers, 8 heads, 16x16 pos-embed
#   |--- Decoder
#   |    ConvT/2 + concat + DDS    : 384 -> 384  (uses skip[3])
#   |    ConvT/2 + concat + DDS    : 384 -> 192  (uses skip[2])  -> aux_deep
#   |    ConvT/2 + concat + DDS    : 192 -> 96   (uses skip[1])  -> aux_shallow
#   |    ConvT/2 + concat + DDS    : 96  -> 48   (uses skip[0])
#   |--- Heads
#        OutConv 1x1 (48  -> C)          main head, full resolution
#        OutConv 1x1 (192 -> C) + up     aux_deep    head (training only, w=0.2)
#        OutConv 1x1 (96  -> C) + up     aux_shallow head (training only, w=0.4)
#
# Total: ~4.7 M params, ~52 GFLOPs at 1×1×512×640 — about 5× smaller and
# ~5× cheaper than the VGG-16 baselines (~24 M params, ~247 GFLOPs).
# ===========================================================================

# Deep-supervision weights for the two aux heads. Shallow head gets a higher
# weight because its features are closer to the final output (more "ready")
# and a stronger gradient there speeds up convergence of the upper decoder.
AUX_W_SHALLOW: Final[float] = 0.4
AUX_W_DEEP:    Final[float] = 0.2
# Width schedule. The first entry is the stem multiplier; the remaining four
# scale the encoder stages. (1, 2, 4, 8, 8) keeps the last two stages at the
# same width — typical for U-Net to avoid an explosion at the bottleneck.
DEFAULT_CHANNEL_MULTIPLIERS: Final[tuple[int, ...]] = (1, 2, 4, 8, 8)


class _OutConv1x1(nn.Module):
    """1×1 conv head — projects feature channels to num_classes logits."""
    def __init__(self, in_channels: int, num_classes: int) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, num_classes, kernel_size=1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class CustomLWIRUNet(nn.Module):
    """From-scratch U-Net for 1-channel LWIR thermal imagery.

    Forward output:
        * training mode + use_aux_heads=True ->
              (main, aux_shallow, aux_deep), three tensors all [B, C, H, W]
        * eval mode (or use_aux_heads=False) ->
              just ``main`` of shape [B, C, H, W]

    This dual return contract lets the trainer compute the combined deep-
    supervision loss during training while keeping inference cheap (eval
    skips the two aux heads entirely).
    """
    def __init__(self, num_classes: int = NUM_CLASSES, in_channels: int = 1,
                 stem_channels: int = 48,
                 channel_multipliers: tuple[int, ...] = DEFAULT_CHANNEL_MULTIPLIERS,
                 transformer_layers: int = 2, transformer_heads: int = 8,
                 use_aux_heads: bool = True, groups_norm: int = 8) -> None:
        super().__init__()
        if in_channels != 1:
            raise ValueError('CustomLWIRUNet is single-channel only')
        # Derive concrete channel widths from the stem and multipliers.
        widths = tuple(stem_channels * m for m in channel_multipliers)
        deepest = widths[-1]
        if deepest % transformer_heads != 0:
            raise ValueError(f'deepest width {deepest} not divisible by {transformer_heads} heads')

        self.num_classes = num_classes
        self.in_channels = in_channels
        self.widths = widths
        self.use_aux_heads = use_aux_heads

        # --- Stem (full resolution). The single-channel input means standard
        # 3×3 convs here, not depthwise-separable (see StandardConvBlock).
        self.stem = nn.Sequential(
            StandardConvBlock(in_channels, widths[0], groups_norm),
            StandardConvBlock(widths[0],   widths[0], groups_norm),
        )
        # --- Encoder: four (MaxPool + DoubleDSConv) stages, each halving
        # spatial dims and expanding channels per the multiplier schedule.
        self.pools = nn.ModuleList([nn.MaxPool2d(2, 2) for _ in range(4)])
        self.encoder_blocks = nn.ModuleList([
            DoubleDSConv(widths[i], widths[i + 1], groups_norm) for i in range(4)
        ])
        # --- Bottleneck: shape-preserving transformer at the deepest stage.
        self.bottleneck = TransformerBottleneck(
            channels=deepest, num_heads=transformer_heads,
            num_layers=transformer_layers, mlp_ratio=2, spatial_size=16,
        )

        # --- Decoder. Skips are consumed deep -> shallow; ConvT halves
        # channels and doubles spatial dims, then we concat the skip and
        # let DoubleDSConv mix back down to the skip's channel count.
        skip_channels = [widths[3], widths[2], widths[1], widths[0]]  # deep -> shallow
        self.up_convs = nn.ModuleList()
        self.decoder_blocks = nn.ModuleList()
        up_in = deepest
        for skip_c in skip_channels:
            self.up_convs.append(nn.ConvTranspose2d(up_in, skip_c, 2, stride=2))
            self.decoder_blocks.append(DoubleDSConv(skip_c * 2, skip_c, groups_norm))
            up_in = skip_c

        # --- Heads.
        # Main head: 1×1 at full input resolution (decoder output width = stem width).
        self.out_conv = _OutConv1x1(widths[0], num_classes)
        # Aux heads attach to decoder stages 2 (deeper, weight 0.2) and 3
        # (shallower, weight 0.4). Their predictions are bilinear-upsampled to
        # the input resolution inside forward() so all three heads share one mask.
        if use_aux_heads:
            self.aux_head_deep    = _OutConv1x1(widths[2], num_classes)  # 192ch
            self.aux_head_shallow = _OutConv1x1(widths[1], num_classes)  # 96ch
        else:
            self.aux_head_deep = self.aux_head_shallow = None

        # Apply Kaiming init across all conv layers (init_silu_weights treats
        # SiLU as leaky_relu(a=0.01)).
        init_silu_weights(self)

    def forward(self, x):
        # --- Encoder pass: stem + 4 (pool, conv) stages -----------------
        s0 = self.stem(x)        # stride 1, widths[0] channels
        skips = [s0]
        feat = s0
        for pool, block in zip(self.pools, self.encoder_blocks):
            feat = block(pool(feat))
            skips.append(feat)
        # After the loop skips = [s0(stride 1), s1(stride 2), s2(stride 4),
        # s3(stride 8), s4(stride 16)]

        # --- Bottleneck (global attention on the deepest feature map) ---
        out = self.bottleneck(skips[-1])

        # --- Decoder pass: deep -> shallow, consuming skips[3..0] -------
        decoder_outs = []
        for up, block, skip in zip(self.up_convs, self.decoder_blocks,
                                   (skips[3], skips[2], skips[1], skips[0])):
            out = up(out)
            # Defensive padding when input H,W aren't divisible by 16 (no-op
            # at the default 512×640 input).
            if out.shape[-2:] != skip.shape[-2:]:
                dh = skip.size(-2) - out.size(-2)
                dw = skip.size(-1) - out.size(-1)
                out = F.pad(out, [dw // 2, dw - dw // 2, dh // 2, dh - dh // 2])
            out = block(torch.cat([skip, out], dim=1))
            decoder_outs.append(out)
        # decoder_outs[0..3] are the four decoder stage outputs in
        # stride 8, 4, 2, 1 order.

        # --- Heads ------------------------------------------------------
        main_logits = self.out_conv(decoder_outs[-1])
        # Aux heads only contribute during training. In eval mode we skip
        # them entirely, making inference byte-identical to a no-aux variant.
        if self.use_aux_heads and self.training and self.aux_head_deep is not None:
            target_size = main_logits.shape[-2:]
            aux_deep    = F.interpolate(self.aux_head_deep(decoder_outs[1]),    size=target_size, mode='bilinear', align_corners=False)
            aux_shallow = F.interpolate(self.aux_head_shallow(decoder_outs[2]), size=target_size, mode='bilinear', align_corners=False)
            return main_logits, aux_shallow, aux_deep
        return main_logits


def build_custom_lwir_unet(**kwargs) -> CustomLWIRUNet:
    """Thin builder for CustomLWIRUNet (kept as a function for API symmetry)."""
    return CustomLWIRUNet(**kwargs)


### 4.4 Loss & metrics

* **Focal loss** (γ=2) — `bridge`, `obstacle`, `living_obs` are severely
  imbalanced (`living_obs` is just ~0.05 % of pixels). Standard CE drowns in
  the easy sky/water majority; focal loss down-weights pixels the model
  already classifies confidently and concentrates gradient on the hard ones.
* **Streaming confusion matrix** — accumulated over the entire validation
  (and final train + test) pass, then per-class **IoU, Precision, Recall**
  are exact `TP / union`, `TP / column_sum`, `TP / row_sum`. The standard
  semantic-seg protocol.
* **`ignore_index=255`** — both the loss and the metric trackers skip border
  pixels left over from rotation augmentation.


In [ ]:
class FocalLoss(nn.Module):
    """Multi-class focal loss for dense prediction.

    Formula (Lin et al. 2017, extended to per-pixel):
        FL(p_t) = - (1 - p_t)^gamma * log(p_t)
    where ``p_t`` is the softmax probability the model assigned to the true
    class at that pixel. Because ``-log(p_t) == CE(logits, target)``, we
    compute pixel-wise CE first and modulate it by ``(1 - p_t)^gamma`` to
    down-weight the easy pixels.

    Why this beats plain CE on MassMIND: classes 0 (sky) and 1 (water)
    together are ~80% of pixels and trivial to classify. With plain CE the
    gradient is dominated by those easy pixels and the model never bothers
    learning the rare classes (``bridge``, ``living_obs``). Focal loss
    shrinks the contribution of confidently-correct pixels (high ``p_t``)
    by a factor of ``(1 - p_t)^gamma``, which at ``gamma=2`` is ~100× for
    p_t = 0.9 and ~10000× for p_t = 0.99 — basically reclaiming the
    gradient budget for the hard pixels.
    """
    def __init__(self, gamma: float = 2.0, alpha: float | None = None,
                 ignore_index: int = 255, reduction: str = 'mean') -> None:
        super().__init__()
        self.gamma = float(gamma)
        self.alpha = alpha             # scalar down-weighting (None = unweighted)
        self.ignore_index = ignore_index
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Pixel-wise CE with ignore_index honoured (ignored pixels return 0).
        ce = F.cross_entropy(logits, targets, reduction='none', ignore_index=self.ignore_index)
        # p_t = exp(-CE) recovers the softmax prob of the true class.
        pt = torch.exp(-ce)
        # The focal modulation: (1 - p_t)^gamma. At gamma=2 this is the standard.
        focal = (1.0 - pt).pow(self.gamma) * ce
        # Optional global alpha (acts like a constant scale on all classes).
        if self.alpha is not None:
            focal = float(self.alpha) * focal
        # Mean over non-ignored pixels, guarding against an all-ignored batch.
        valid = targets != self.ignore_index
        if self.reduction == 'sum':
            return focal[valid].sum()
        n = valid.sum()
        if n == 0:
            return focal.sum() * 0.0
        return focal[valid].sum() / n


class ConfusionMatrixTracker:
    """Streaming K×K confusion matrix; rows = ground truth, cols = predictions.

    Why a confusion matrix and not per-batch IoU averaging:

      * Per-batch IoU is **biased** when batches are small or class-
        imbalanced. Rare classes get a noisy IoU per batch that destabilises
        the mean.
      * A single CM accumulated over the entire validation set gives the
        exact dataset-level IoU via ``TP / (TP + FP + FN)``. This is the
        standard for semantic segmentation benchmarks (Cityscapes, ADE20K,
        the MassMIND paper).

    Memory cost: K×K int64 -> 392 bytes for 7 classes. Effectively free.
    """
    def __init__(self, num_classes: int, ignore_index: int = 255) -> None:
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.cm = torch.zeros(num_classes, num_classes, dtype=torch.int64)

    @torch.no_grad()
    def update(self, pred: torch.Tensor, target: torch.Tensor) -> None:
        """Add one batch to the running confusion matrix.

        Vectorised: encode (target, pred) as a single index in [0, K*K),
        bincount it, then reshape to K×K. Roughly 100× faster than a
        Python double loop and matches the standard idiom used in
        torchvision / mmseg.
        """
        pred = pred.detach().cpu().reshape(-1)
        target = target.detach().cpu().reshape(-1)
        valid = target != self.ignore_index
        # Clamp predictions defensively (they shouldn't be OOB after argmax
        # over K logits, but the clamp is cheap insurance).
        pred = pred[valid].clamp_(0, self.num_classes - 1)
        target = target[valid]
        idx = target * self.num_classes + pred
        binc = torch.bincount(idx, minlength=self.num_classes ** 2)
        self.cm += binc.reshape(self.num_classes, self.num_classes)

    def per_class_iou(self) -> torch.Tensor:
        """IoU = TP / (TP + FP + FN) per class. NaN if a class never appeared."""
        cm = self.cm.float()
        tp = cm.diag()              # diagonal: true positives per class
        fp = cm.sum(0) - tp         # column sum minus diag = false positives
        fn = cm.sum(1) - tp         # row sum minus diag    = false negatives
        union = tp + fp + fn
        return torch.where(union > 0, tp / union, torch.full_like(tp, float('nan')))

    def mean_iou(self) -> float:
        """mIoU = mean of per-class IoU, ignoring classes with NaN IoU."""
        iou = self.per_class_iou()
        finite = iou[~torch.isnan(iou)]
        return float(finite.mean()) if finite.numel() > 0 else float('nan')

    def pixel_accuracy(self) -> float:
        """Fraction of (non-ignored) pixels classified correctly."""
        total = self.cm.sum()
        return float(self.cm.diag().sum() / total) if total > 0 else float('nan')


In [ ]:
# ===========================================================================
# Per-class IoU / Precision / Recall / F1 from a confusion matrix
# ===========================================================================
# All four follow directly from the CM. cm[i, j] = number of pixels with true
# class i predicted as class j. Then:
#   TP_c    = cm[c, c]                       (diagonal)
#   col_c   = cm[:, c].sum()  -> TP_c + FP_c
#   row_c   = cm[c, :].sum()  -> TP_c + FN_c
#   union_c = col_c + row_c - TP_c           (TP + FP + FN)
#
# Precision_c = TP / (TP + FP)   = TP / col_c
# Recall_c    = TP / (TP + FN)   = TP / row_c
# IoU_c       = TP / (TP + FP + FN)
# F1_c        = 2 * P * R / (P + R)


def _as_float_cm(cm):
    """Accept a torch.Tensor or list-of-lists CM; return a float tensor."""
    return cm.float() if isinstance(cm, torch.Tensor) else torch.tensor(cm).float()


def per_class_iou_from_cm(cm) -> torch.Tensor:
    """IoU per class. NaN when a class never appeared in ground truth or
    prediction (TP + FP + FN = 0)."""
    cm = _as_float_cm(cm)
    tp = cm.diag()
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    union = tp + fp + fn
    return torch.where(union > 0, tp / union, torch.full_like(tp, float('nan')))


def per_class_precision_from_cm(cm) -> torch.Tensor:
    """Precision per class. NaN when the class was never predicted (col_sum=0)."""
    cm = _as_float_cm(cm)
    tp = cm.diag()
    col = cm.sum(0)
    return torch.where(col > 0, tp / col, torch.full_like(tp, float('nan')))


def per_class_recall_from_cm(cm) -> torch.Tensor:
    """Recall per class. NaN when the class never appeared in ground truth
    (row_sum=0)."""
    cm = _as_float_cm(cm)
    tp = cm.diag()
    row = cm.sum(1)
    return torch.where(row > 0, tp / row, torch.full_like(tp, float('nan')))


def per_class_f1_from_cm(cm) -> torch.Tensor:
    """F1 per class: F1 = 2·TP / (2·TP + FP + FN)."""
    cm = _as_float_cm(cm)
    tp = cm.diag()
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    denom = 2 * tp + fp + fn
    return torch.where(denom > 0, 2 * tp / denom, torch.full_like(tp, float('nan')))


### 4.5 Training loop

Identical scaffolding to notebook 07 — AdamW (lr 1e-4), focal loss, cosine
schedule with optional 5 % linear warmup for the from-scratch configs, AMP
on CUDA. Three differences worth flagging:

* `TrainConfig` carries the **`augment`** flag, threaded into the dataset.
* The trainer calls **`train_ds.set_epoch(epoch)`** at the top of every
  epoch so the deterministic augmentation advances.
* On every validated epoch (1, 5, 10, …, 35) the trainer **also evaluates
  on the full training set** (no augmentation, no shuffle). The resulting
  `train_mIoU` is logged alongside `val_mIoU` so the convergence plot can
  show the train-vs-val gap that signals overfitting. Cost: ~70 s extra per
  validated epoch (~6 minutes total over the 6 runs).
* At the end of every run, **`evaluate_final`** is called automatically: it
  reloads the best-by-val checkpoint and computes the **full-train** and
  **full-test** confusion matrices, saved to `final_eval.json` next to
  `metrics.csv`. The results section reads these to report per-class IoU,
  Precision and Recall on both splits.


In [ ]:
# ===========================================================================
# TrainConfig
# ===========================================================================
@dataclass
class TrainConfig:
    name: str                       # run label / output subdir
    model: str                      # 'vgg16' or 'custom_lwir'
    encoder_weights: str | None     # 'imagenet' or None
    augment: bool                   # geometric augmentation on/off
    epochs: int = 35
    batch_size: int = 4
    lr: float = 1e-4
    weight_decay: float = 1e-4
    focal_gamma: float = 2.0
    focal_alpha: float | None = None
    seed: int = 42
    warmup_frac: float = 0.0
    stem_channels: int = 48
    transformer_layers: int = 2
    use_aux_heads: bool = True
    amp: bool = True

VAL_EVERY = 5   # validate (and evaluate on full train set) on epoch 1, every 5th, and the last


def build_model_from_cfg(cfg: TrainConfig) -> nn.Module:
    """Dispatch to the right model builder."""
    if cfg.model == 'vgg16':
        return build_unet_vgg16(num_classes=NUM_CLASSES, in_channels=1,
                                encoder_weights=cfg.encoder_weights)
    if cfg.model == 'custom_lwir':
        return build_custom_lwir_unet(
            num_classes=NUM_CLASSES, in_channels=1,
            stem_channels=cfg.stem_channels,
            transformer_layers=cfg.transformer_layers,
            use_aux_heads=cfg.use_aux_heads,
        )
    raise ValueError(cfg.model)


def _compute_loss(output, masks, criterion) -> torch.Tensor:
    """Combine main + aux-head losses (L_main + 0.4·L_shallow + 0.2·L_deep)."""
    if isinstance(output, tuple):
        main, aux_shallow, aux_deep = output
        return (criterion(main, masks)
                + AUX_W_SHALLOW * criterion(aux_shallow, masks)
                + AUX_W_DEEP    * criterion(aux_deep,    masks))
    return criterion(output, masks)


def train_one_epoch(model, loader, criterion, optimizer, device, *,
                    amp: bool, scheduler=None, step_per_batch: bool = False) -> float:
    """One training epoch. Returns mean per-sample loss."""
    model.train()
    use_amp = amp and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    running, n = 0.0, 0
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        masks  = batch['mask'].to(device,  non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=use_amp):
            out = model(images)
            loss = _compute_loss(out, masks, criterion)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None and step_per_batch:
            scheduler.step()
        bs = images.size(0)
        running += float(loss.detach()) * bs
        n += bs
    return running / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, device, *, amp: bool):
    """One validation pass. Returns (mean loss, ConfusionMatrixTracker).

    Forward + loss run under autocast (CUDA only); argmax + CM update stay
    fp32 so the metric is unaffected by autocast precision.
    """
    model.eval()
    use_amp = amp and device.type == 'cuda'
    tracker = ConfusionMatrixTracker(NUM_CLASSES, ignore_index=MASK_IGNORE_INDEX)
    running, n = 0.0, 0
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        masks  = batch['mask'].to(device,  non_blocking=True)
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, masks)
        tracker.update(logits.argmax(1), masks)
        bs = images.size(0)
        running += float(loss.detach()) * bs
        n += bs
    return running / max(n, 1), tracker


def build_scheduler(optimizer, cfg: TrainConfig, steps_per_epoch: int):
    """Cosine schedule, optionally preceded by a linear warmup."""
    if cfg.warmup_frac <= 0:
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs), False
    total_steps = max(cfg.epochs * steps_per_epoch, 1)
    warmup_steps = max(int(round(cfg.warmup_frac * total_steps)), 1)
    cosine_steps = max(total_steps - warmup_steps, 1)
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-3,
                                               end_factor=1.0, total_iters=warmup_steps)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cosine_steps)
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps]), True


@torch.no_grad()
def evaluate_final(cfg: TrainConfig, out_dir: Path) -> dict:
    """End-of-run evaluation. Reloads the best-by-val checkpoint and runs one
    pass over the FULL training set and one over the FULL test set, saving
    both confusion matrices to ``final_eval.json``.

    Returns the dict {'cm_train': [...], 'cm_test': [...], 'best_epoch': int}.
    Idempotent — if final_eval.json already exists, just loads and returns it.
    """
    final_path = out_dir / 'final_eval.json'
    if final_path.exists():
        return json.loads(final_path.read_text())

    ckpt = torch.load(out_dir / 'checkpoint_best.pt', map_location=DEVICE, weights_only=False)
    model = build_model_from_cfg(cfg).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    splits = json.loads(SPLIT_PATH.read_text())['splits']
    mean, std = ckpt['mean'], ckpt['std']
    pin = torch.cuda.is_available()
    nw = 2 if torch.cuda.is_available() else 0

    def _cm_for(filenames: list[str]) -> torch.Tensor:
        ds = MassMINDDataset(DATA_ROOT, filenames, mean, std,
                             downsample_hw=DOWNSAMPLE_HW, augment=False)
        loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=nw, pin_memory=pin, collate_fn=collate_fn)
        tr = ConfusionMatrixTracker(NUM_CLASSES, ignore_index=MASK_IGNORE_INDEX)
        for batch in loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            masks  = batch['mask'].to(DEVICE,  non_blocking=True)
            logits = model(images)
            tr.update(logits.argmax(1), masks)
        return tr.cm

    cm_train = _cm_for(splits['train'])
    cm_test  = _cm_for(splits['test'])
    del model

    payload = {
        'cm_train': cm_train.tolist(),
        'cm_test':  cm_test.tolist(),
        'best_epoch': int(ckpt['epoch']),
    }
    final_path.write_text(json.dumps(payload))
    return payload


def run_training(cfg: TrainConfig, output_dir: Path) -> Path:
    """Train one config end-to-end. Writes config.json, metrics.csv,
    checkpoint_best/last.pt, and (via evaluate_final at the end) final_eval.json.
    """
    torch.manual_seed(cfg.seed)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / 'config.json').write_text(json.dumps(asdict(cfg), indent=2))

    # --- Data. Three datasets: train (with cfg.augment), train_eval (no
    # augmentation, used for the full-train-set evaluation pass), val. -----
    splits = json.loads(SPLIT_PATH.read_text())['splits']
    stats  = json.loads(STATS_PATH.read_text())
    mean, std = stats['mean'], stats['std']
    train_ds      = MassMINDDataset(DATA_ROOT, splits['train'], mean, std,
                                    downsample_hw=DOWNSAMPLE_HW, augment=cfg.augment)
    train_eval_ds = MassMINDDataset(DATA_ROOT, splits['train'], mean, std,
                                    downsample_hw=DOWNSAMPLE_HW, augment=False)
    val_ds        = MassMINDDataset(DATA_ROOT, splits['val'], mean, std,
                                    downsample_hw=DOWNSAMPLE_HW, augment=False)

    pin = torch.cuda.is_available()
    nw = 2 if torch.cuda.is_available() else 0
    train_loader      = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                                   num_workers=nw, pin_memory=pin, collate_fn=collate_fn,
                                   persistent_workers=nw > 0)
    train_eval_loader = DataLoader(train_eval_ds, batch_size=cfg.batch_size, shuffle=False,
                                   num_workers=nw, pin_memory=pin, collate_fn=collate_fn,
                                   persistent_workers=nw > 0)
    val_loader        = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                                   num_workers=nw, pin_memory=pin, collate_fn=collate_fn,
                                   persistent_workers=nw > 0)

    device = DEVICE
    model = build_model_from_cfg(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'[{cfg.name}] params: {n_params/1e6:.2f} M  |  augment: {cfg.augment}  '
          f'|  device: {device}  |  train batches: {len(train_loader)}  '
          f'val batches: {len(val_loader)}')

    criterion = FocalLoss(gamma=cfg.focal_gamma, alpha=cfg.focal_alpha,
                          ignore_index=MASK_IGNORE_INDEX)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler, step_per_batch = build_scheduler(optimizer, cfg, len(train_loader))

    fieldnames = (['epoch', 'train_loss', 'val_loss',
                   'train_mIoU', 'val_mIoU', 'pixel_acc']
                  + [f'iou_{n}' for n in CLASS_NAMES] + ['lr', 'elapsed_s'])
    csv_path = output_dir / 'metrics.csv'
    with csv_path.open('w', newline='') as f:
        csv.writer(f).writerow(fieldnames)

    best_miou = -1.0
    run_t0 = time.time()
    # tqdm progress bars don't render in Kaggle's committed (papermill) log,
    # so we print one explicit line per epoch — train_mIoU, val_mIoU, ETA.
    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        train_ds.set_epoch(epoch)
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            amp=cfg.amp, scheduler=scheduler, step_per_batch=step_per_batch,
        )

        # Validate on epoch 1, every VAL_EVERY epochs, and the final epoch.
        # On those same epochs, also evaluate on the full TRAINING set (no
        # augmentation, no shuffle) to track the train/val gap.
        do_val = (epoch == 1) or (epoch % VAL_EVERY == 0) or (epoch == cfg.epochs)
        if do_val:
            val_loss,   val_tracker   = evaluate(model, val_loader,        criterion, device, amp=cfg.amp)
            _,          train_tracker = evaluate(model, train_eval_loader, criterion, device, amp=cfg.amp)
            val_miou   = val_tracker.mean_iou()
            train_miou = train_tracker.mean_iou()
            pixel_acc  = val_tracker.pixel_accuracy()
            per_class  = val_tracker.per_class_iou().tolist()
        else:
            val_loss = val_miou = train_miou = pixel_acc = float('nan')
            per_class = [float('nan')] * NUM_CLASSES

        if not step_per_batch:
            scheduler.step()
        elapsed = time.time() - t0

        row = {'epoch': epoch, 'train_loss': round(train_loss, 5),
               'val_loss':   round(val_loss,   5) if val_loss   == val_loss   else float('nan'),
               'train_mIoU': round(train_miou, 5) if train_miou == train_miou else float('nan'),
               'val_mIoU':   round(val_miou,   5) if val_miou   == val_miou   else float('nan'),
               'pixel_acc':  round(pixel_acc,  5) if pixel_acc  == pixel_acc  else float('nan'),
               'lr': optimizer.param_groups[0]['lr'], 'elapsed_s': round(elapsed, 1)}
        for name, iou in zip(CLASS_NAMES, per_class):
            row[f'iou_{name}'] = round(iou, 5) if iou == iou else float('nan')
        with csv_path.open('a', newline='') as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row)

        # Per-epoch progress line (flush=True so it appears in Kaggle's log live).
        avg_epoch = (time.time() - run_t0) / epoch
        eta_min = avg_epoch * (cfg.epochs - epoch) / 60.0
        if do_val:
            val_str = f'train_mIoU={train_miou:.4f}  val_mIoU={val_miou:.4f}'
        else:
            val_str = 'val skipped'.ljust(36)
        print(f'[{cfg.name}] epoch {epoch:2d}/{cfg.epochs}  '
              f'train_loss={train_loss:.4f}  {val_str}  '
              f'| {elapsed:5.0f}s/epoch  ETA {eta_min:5.1f} min', flush=True)

        # Best-by-val checkpoint.
        if do_val and val_miou > best_miou and val_miou == val_miou:
            best_miou = val_miou
            torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                        'mean': mean, 'std': std, 'config': asdict(cfg)},
                       output_dir / 'checkpoint_best.pt')

    torch.save({'epoch': cfg.epochs, 'model_state': model.state_dict(),
                'mean': mean, 'std': std, 'config': asdict(cfg)},
               output_dir / 'checkpoint_last.pt')
    print(f'[{cfg.name}] training complete. best val mIoU = {best_miou:.4f}')

    # End-of-run final evaluation on the full train and test sets.
    print(f'[{cfg.name}] running final train + test evaluation...')
    evaluate_final(cfg, output_dir)
    print(f'[{cfg.name}] final_eval.json written.', flush=True)
    return output_dir


## 5. Training runs — the 3 × 2 matrix

Six runs: each of the three models trained **with** and **without**
augmentation, **35 epochs** each at 384×480. Estimated ~7.5–8 h total on a
Kaggle T4 — fits one session with margin.

| | no augmentation | with augmentation |
|---|---|---|
| `vgg16_pretrained` | run 1 | run 2 |
| `vgg16_scratch` | run 3 | run 4 |
| `custom_lwir` | run 5 | run 6 |

The training cell is **resumable**: any config whose `metrics.csv` already
exists is skipped, so re-running after a Kaggle session timeout continues
where it stopped. `evaluate_final` is also idempotent — it short-circuits if
`final_eval.json` already exists.


In [ ]:
EPOCHS = 35

# 3 models × {no-aug, aug}. Run names are "<model>__<augcond>" — the double
# underscore is the split point used by the analysis cells.
_MODELS = [
    ('vgg16_pretrained', 'vgg16',       'imagenet'),
    ('vgg16_scratch',    'vgg16',       None),
    ('custom_lwir',      'custom_lwir', None),
]

CONFIGS = []
for label, model_name, weights in _MODELS:
    for aug_label, augment in [('noaug', False), ('aug', True)]:
        CONFIGS.append(TrainConfig(
            name=f'{label}__{aug_label}',
            model=model_name,
            encoder_weights=weights,
            augment=augment,
            epochs=EPOCHS,
            # 5 % linear warmup for the from-scratch configs (noisier early
            # gradients); pure cosine for the pretrained encoder.
            warmup_frac=0.0 if weights == 'imagenet' else 0.05,
        ))

RUN_DIRS = {c.name: RUNS_DIR / c.name for c in CONFIGS}
for c in CONFIGS:
    print(f'  {c.name:28s}  model={c.model:12s}  '
          f'weights={str(c.encoder_weights):8s}  augment={c.augment}')


In [ ]:
# Train all six configs sequentially. Resumable — skips any config whose
# metrics.csv already exists. Each run also writes final_eval.json
# (full-train and full-test confusion matrices on the best checkpoint).
for cfg in CONFIGS:
    out = RUN_DIRS[cfg.name]
    if (out / 'metrics.csv').exists() and (out / 'final_eval.json').exists():
        print(f'[{cfg.name}] already trained + evaluated — skipping.')
        continue
    if (out / 'metrics.csv').exists() and (out / 'checkpoint_best.pt').exists():
        # Training already done but final_eval missing — just run that.
        print(f'[{cfg.name}] training present; running final evaluation only.')
        evaluate_final(cfg, out)
        continue
    run_training(cfg, out)
print('\nAll training runs complete.')


## 6. Results

All tables and plots are saved under `runs_final/analysis/` as CSV + PNG + PDF.
The CMs used for the per-class IoU / Precision / Recall numbers come from
`final_eval.json` — written at the end of each training run on the
**held-out test set** (291 frames) and the **full training set** (2042 frames).


### 6.1 Load all runs


In [ ]:
# Forward-pass GFLOPs measured at the training/eval resolution (384x480).
# Params are resolution-independent.
def params_and_flops_for(cfg: TrainConfig, hw: tuple[int, int] = DOWNSAMPLE_HW) -> tuple[float, float]:
    """(params_M, gflops) for one forward pass at 1x1xHxW in eval mode."""
    model = build_model_from_cfg(cfg).eval()
    n_params = sum(p.numel() for p in model.parameters())
    x = torch.zeros(1, 1, hw[0], hw[1])
    counter = FlopCounterMode(display=False)
    with counter, torch.no_grad():
        model(x)
    n_flops = counter.get_total_flops()
    del model
    return n_params / 1e6, n_flops / 1e9


# Load every run's metrics.csv + final_eval.json.
runs_info = []
for cfg in CONFIGS:
    out = RUN_DIRS[cfg.name]
    metrics_path = out / 'metrics.csv'
    final_path = out / 'final_eval.json'
    if not metrics_path.exists():
        print(f'  WARN: no metrics.csv for {cfg.name} — skipping')
        continue
    df = pd.read_csv(metrics_path)
    final = json.loads(final_path.read_text()) if final_path.exists() else None
    params_m, gflops = params_and_flops_for(cfg)
    model_label, aug_cond = cfg.name.split('__')
    cm_train = torch.tensor(final['cm_train']) if final else None
    cm_test  = torch.tensor(final['cm_test'])  if final else None
    runs_info.append({'cfg': cfg, 'df': df, 'out': out,
                      'model': model_label, 'aug': aug_cond,
                      'params_M': params_m, 'gflops': gflops,
                      'cm_train': cm_train, 'cm_test': cm_test,
                      'best_epoch': final['best_epoch'] if final else None})

print(f'Loaded {len(runs_info)} of {len(CONFIGS)} runs:')
for r in runs_info:
    val_best = r['df']['val_mIoU'].max()
    cm_msg = 'OK' if r['cm_test'] is not None else 'MISSING final_eval.json'
    print(f"  {r['cfg'].name:28s}  best val_mIoU={val_best:.4f}  "
          f"params={r['params_M']:.2f}M  GFLOPs={r['gflops']:.2f}  final_eval={cm_msg}")


### 6.2 Convergence curves

All six runs on three shared axes — validation mIoU, training mIoU, training
loss. Colour encodes the architecture, line style the augmentation condition
(solid = with augmentation, dashed = no augmentation). The train mIoU panel
sits above the val curve by the train/val gap, which widens with overfitting
and should narrow under augmentation.


In [ ]:
# Combined view — all 6 runs on three shared axes (val mIoU, train mIoU,
# train loss). Colour = architecture, line style = augmentation.
model_list = sorted({r['model'] for r in runs_info})
conv_colors = {m: c for m, c in zip(model_list, plt.cm.tab10.colors)}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for r in runs_info:
    style = '-' if r['aug'] == 'aug' else '--'
    col = conv_colors[r['model']]
    lbl = f"{r['model']} ({'aug' if r['aug'] == 'aug' else 'no aug'})"
    # Val mIoU and train mIoU only exist on validated epochs (NaN otherwise).
    df_v = r['df'].dropna(subset=['val_mIoU'])
    df_t = r['df'].dropna(subset=['train_mIoU'])
    axes[0].plot(df_v['epoch'], df_v['val_mIoU'],   style, marker='o', ms=4, color=col, label=lbl)
    axes[1].plot(df_t['epoch'], df_t['train_mIoU'], style, marker='o', ms=4, color=col, label=lbl)
    # Train loss is logged every epoch.
    axes[2].plot(r['df']['epoch'], r['df']['train_loss'], style, marker='.', ms=3, color=col, label=lbl)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val mIoU');   axes[0].set_title('Validation mIoU')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Train mIoU'); axes[1].set_title('Training mIoU (full train set)')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Train loss'); axes[2].set_title('Training loss')
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(fontsize=7, loc='best')
fig.suptitle('Convergence — all six runs')
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'convergence_all.{ext}', dpi=150)
plt.show()


### 6.3 Final results table

One row per run. Headline metrics from the **best-by-val** checkpoint:
parameter count and forward-pass GFLOPs for model complexity, full-set IoU
on both training and test splits, **macro-mean Precision** and **macro-mean
Recall** on the test set (averaged across all 7 classes — so the
0.05 %-pixel `living_obs` class counts equally with `water`, which keeps
the metric honest on imbalanced data), and wall-clock training time.

Per-class breakdowns for both splits are in §6.4.


In [ ]:
def _macro_mean(metric_tensor):
    """Mean across classes, ignoring NaN — same semantics as mIoU."""
    finite = metric_tensor[~torch.isnan(metric_tensor)]
    return float(finite.mean()) if finite.numel() > 0 else float('nan')


rows = []
for r in runs_info:
    if r['cm_test'] is None:
        continue
    df = r['df']
    train_mIoU  = _macro_mean(per_class_iou_from_cm(r['cm_train']))
    test_mIoU   = _macro_mean(per_class_iou_from_cm(r['cm_test']))
    mean_test_P = _macro_mean(per_class_precision_from_cm(r['cm_test']))
    mean_test_R = _macro_mean(per_class_recall_from_cm(r['cm_test']))
    rows.append({
        'model':       r['model'],
        'aug':         r['aug'],
        'params_M':    round(r['params_M'], 2),
        'gflops':      round(r['gflops'],   2),
        'train_mIoU':  round(train_mIoU,    4),
        'test_mIoU':   round(test_mIoU,     4),
        'mean_P_test': round(mean_test_P,   4),
        'mean_R_test': round(mean_test_R,   4),
        'train_min':   round(df['elapsed_s'].sum() / 60.0, 1),
    })

comparison_table = pd.DataFrame(rows).sort_values(['model', 'aug']).reset_index(drop=True)
comparison_table.to_csv(ANALYSIS_DIR / 'final_results.csv', index=False)
comparison_table


### 6.4 Per-class IoU, Precision and Recall (train and test)

The three core metrics required by the assignment, on the full training set
and the held-out test set, broken down by class. Each table is indexed by
*(run, split)* — two adjacent rows for the same run let you read the
train-vs-test gap per class at a glance.

`NaN` means the class was never predicted (Precision) or never appeared in
the ground truth (Recall) for that run.


In [ ]:
def _metric_table(metric_fn, label: str) -> pd.DataFrame:
    """Multi-index (run, split) × class table for one CM-derived metric."""
    rows = []
    for r in runs_info:
        if r['cm_test'] is None or r['cm_train'] is None:
            continue
        for split, cm in [('train', r['cm_train']), ('test', r['cm_test'])]:
            vals = metric_fn(cm).tolist()
            row = {'run': r['cfg'].name, 'split': split}
            for c, v in zip(CLASS_NAMES, vals):
                row[c] = round(v, 4) if v == v else float('nan')
            finite = [v for v in vals if v == v]
            row['mean'] = round(float(np.mean(finite)), 4) if finite else float('nan')
            rows.append(row)
    df = pd.DataFrame(rows).set_index(['run', 'split'])
    df.to_csv(ANALYSIS_DIR / f'per_class_{label}.csv')
    return df


print('=== Per-class IoU (train and test) ===')
display(_metric_table(per_class_iou_from_cm, 'iou'))

print('\n=== Per-class Precision (train and test) ===')
display(_metric_table(per_class_precision_from_cm, 'precision'))

print('\n=== Per-class Recall (train and test) ===')
display(_metric_table(per_class_recall_from_cm, 'recall'))


**Visualisation** — per-class test IoU, one panel per model, no-aug vs aug.
The same data as the IoU table above, in plot form.


In [ ]:
# Visualisation: per-class IoU bars, no-aug vs aug, one panel per model.
# (The same data as the IoU table above, in plot form — the easiest way to
# see which classes augmentation actually moves.)
classes = CLASS_NAMES
models = sorted({r['model'] for r in runs_info if r['cm_test'] is not None})
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4.5), sharey=True)
if len(models) == 1:
    axes = [axes]
x = np.arange(len(classes))
w = 0.38
for ax, model in zip(axes, models):
    for sign, aug_cond, lbl in [(-1, 'noaug', 'no aug'), (1, 'aug', 'with aug')]:
        match = [r for r in runs_info
                 if r['model'] == model and r['aug'] == aug_cond and r['cm_test'] is not None]
        if not match:
            continue
        vals = per_class_iou_from_cm(match[0]['cm_test']).tolist()
        ax.bar(x + sign * w / 2, vals, w, label=lbl)
    ax.set_title(model)
    ax.set_xticks(x); ax.set_xticklabels(classes, rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, 1); ax.grid(alpha=0.3, axis='y'); ax.legend(fontsize=8)
axes[0].set_ylabel('Test IoU')
fig.suptitle('Per-class test IoU: no augmentation vs with augmentation')
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'per_class_test_iou.{ext}', dpi=150)
plt.show()


### 6.5 Augmentation effect

Does geometric augmentation help? The bar chart pairs each model's no-aug
and **test** mIoU under augmentation; the annotation is the delta in
percentage points. Augmentation typically helps the from-scratch models
more than the ImageNet-pretrained one — pretrained features already carry
strong priors that augmentation would otherwise have to teach.


In [ ]:
# Pivot to model x {noaug, aug} on test mIoU and compute the delta.
pivot = comparison_table.pivot(index='model', columns='aug', values='test_mIoU')
pivot = pivot[['noaug', 'aug']]
pivot['delta_pp'] = ((pivot['aug'] - pivot['noaug']) * 100).round(2)
print(pivot.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
models = pivot.index.tolist()
x = np.arange(len(models))
w = 0.38
ax.bar(x - w / 2, pivot['noaug'], w, label='no augmentation')
ax.bar(x + w / 2, pivot['aug'],   w, label='with augmentation')
for i, m in enumerate(models):
    top = max(pivot.loc[m, 'noaug'], pivot.loc[m, 'aug'])
    ax.annotate(f"{pivot.loc[m, 'delta_pp']:+.1f} pp", (i, top + 0.012),
                ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_ylabel('Test mIoU'); ax.set_title('Augmentation effect on test mIoU')
ax.set_ylim(0, 1); ax.grid(alpha=0.3, axis='y'); ax.legend()
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'aug_effect.{ext}', dpi=150)
plt.show()


### 6.6 Per-class confusion

Test-set confusion matrices, a 3×2 grid (model × augmentation condition).
Rows are ground truth, columns predictions; row-normalised so a cell reads
"fraction of true class *i* predicted as class *j*". Off-diagonal mass
shows which classes each model swaps; comparing the no-aug and aug column
for a model shows whether augmentation tightened the diagonal.


In [ ]:
models = sorted({r['model'] for r in runs_info if r['cm_test'] is not None})
aug_order = [('noaug', 'no aug'), ('aug', 'with aug')]

fig, axes = plt.subplots(len(models), len(aug_order),
                         figsize=(5.0 * len(aug_order), 4.3 * len(models)),
                         squeeze=False)
im = None
for ri, model in enumerate(models):
    for ci, (aug_cond, aug_title) in enumerate(aug_order):
        ax = axes[ri][ci]
        match = [r for r in runs_info
                 if r['model'] == model and r['aug'] == aug_cond and r['cm_test'] is not None]
        if not match:
            ax.axis('off'); continue
        cm = match[0]['cm_test'].float()
        cm_norm = cm / cm.sum(dim=1, keepdim=True).clamp(min=1)
        im = ax.imshow(cm_norm.numpy(), cmap='Blues', vmin=0, vmax=1)
        ax.set_title(f'{model} — {aug_title}', fontsize=10)
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right', fontsize=7)
        ax.set_yticks(range(NUM_CLASSES))
        ax.set_yticklabels(CLASS_NAMES, fontsize=7)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                v = cm_norm[i, j].item()
                if v > 0.01:
                    ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6,
                            color='white' if v > 0.5 else 'black')
if im is not None:
    fig.colorbar(im, ax=axes, shrink=0.6, label='Row-normalised confusion')
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'confusion_test.{ext}', dpi=150, bbox_inches='tight')
plt.show()


### 6.7 Pareto — model complexity vs accuracy

Test mIoU against parameter count. Architecture cost is identical across a
model's no-aug and aug runs, so each architecture contributes two points
at the same x. Up-and-to-the-left dominates.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for model in model_list:
    sub = comparison_table[comparison_table['model'] == model].sort_values('aug')
    ax.plot(sub['params_M'], sub['test_mIoU'], '-', color=conv_colors[model], alpha=0.4, zorder=1)
    for _, row in sub.iterrows():
        mk = 'o' if row['aug'] == 'aug' else 's'
        ax.scatter(row['params_M'], row['test_mIoU'], s=130, marker=mk,
                   color=conv_colors[model], edgecolor='black', linewidth=0.6, zorder=2)
        ax.annotate(f"{row['model']}\n({row['aug']})",
                    (row['params_M'], row['test_mIoU']),
                    xytext=(7, 4), textcoords='offset points', fontsize=8)
ax.set_xlabel('Parameters (M)'); ax.set_ylabel('Test mIoU')
ax.set_title('Pareto: parameter count vs test mIoU')
ax.grid(alpha=0.3)
L = plt.matplotlib.lines.Line2D
ax.legend(handles=[
    L([], [], marker='s', linestyle='', color='gray', markeredgecolor='black', label='no augmentation'),
    L([], [], marker='o', linestyle='', color='gray', markeredgecolor='black', label='with augmentation'),
], loc='lower right', fontsize=9, title='augmentation')
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'pareto.{ext}', dpi=150)
plt.show()


### 6.8 Qualitative predictions

Four held-out **test**-set frames: LWIR input, ground truth, and the
`argmax` prediction of each augmentation-trained model. Colormap aligned
across all panels so mis-classifications are visible at a glance.


In [ ]:
@torch.no_grad()
def predict_argmax(cfg: TrainConfig, out_dir: Path, image_t: torch.Tensor) -> torch.Tensor:
    """Reload the best checkpoint, return the [H, W] argmax prediction."""
    ckpt = torch.load(out_dir / 'checkpoint_best.pt', map_location=DEVICE, weights_only=False)
    model = build_model_from_cfg(cfg).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    pred = model(image_t.unsqueeze(0).to(DEVICE)).argmax(1).squeeze(0).cpu()
    del model
    return pred


splits = json.loads(SPLIT_PATH.read_text())['splits']
stats  = json.loads(STATS_PATH.read_text())
test_ds = MassMINDDataset(DATA_ROOT, splits['test'], stats['mean'], stats['std'],
                          downsample_hw=DOWNSAMPLE_HW, augment=False)

rng = np.random.default_rng(seed=0)
sample_idx = sorted(rng.choice(len(test_ds), size=4, replace=False))

aug_runs = [r for r in runs_info if r['aug'] == 'aug']
cmap = plt.get_cmap('tab10', NUM_CLASSES)
ncols = 2 + len(aug_runs)
fig, axes = plt.subplots(len(sample_idx), ncols, figsize=(3.2 * ncols, 3.2 * len(sample_idx)))
for row_i, idx in enumerate(sample_idx):
    sample = test_ds[int(idx)]
    img, mask = sample['image'], sample['mask']
    img_disp = (img.squeeze(0) * stats['std'] + stats['mean']).clamp(0, 1).numpy()
    axes[row_i, 0].imshow(img_disp, cmap='gray')
    axes[row_i, 0].set_title('LWIR input' if row_i == 0 else '')
    axes[row_i, 1].imshow(mask.numpy(), cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1, interpolation='nearest')
    axes[row_i, 1].set_title('Ground truth' if row_i == 0 else '')
    for col_i, r in enumerate(aug_runs, start=2):
        pred = predict_argmax(r['cfg'], r['out'], img)
        axes[row_i, col_i].imshow(pred.numpy(), cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1, interpolation='nearest')
        axes[row_i, col_i].set_title(f"{r['model']} (aug)" if row_i == 0 else '')
    for ax in axes[row_i]:
        ax.axis('off')

patches = [plt.matplotlib.patches.Patch(color=cmap(i), label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=NUM_CLASSES, bbox_to_anchor=(0.5, -0.02), fontsize=9)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'qualitative_test.{ext}', dpi=150, bbox_inches='tight')
plt.show()


## 7. Discussion

The discussion below ties our results back to the MassMIND paper
(Nirgudkar et al., *IJRR* 42(1–2), 2023). The paper reports per-class
Precision/Recall/F1 on the same dataset at thresholds τ = 0.6 and
τ = 0.3 (their Table 7); we evaluate at the standard `argmax` decoding on
the held-out test set.

### Methodology — what we copied and what we changed

Our setup matches the paper on every controllable axis except loss, model
choice, and resolution:

| Aspect | MassMIND paper | This work |
|---|---|---|
| Split | 70 / 20 / 10 train / val / test | **same** |
| Augmentation | rotation ±2°/±5°/±7° + horizontal mirror; brightness jitter explicitly *excluded* (Sec. 5.1) | **same set, but** deterministic per-*(image, epoch)* schedule + on-the-fly (not the paper's offline ×13 expansion) |
| Image size | native 640×512 | **downsampled to 384×480** with anti-aliased `INTER_AREA` to fit Kaggle's 9 h session |
| Epochs | 50 from scratch | **35**, with focal loss compensating the shorter schedule |
| Loss | sparse categorical cross-entropy | **focal loss (γ = 2)** — directly targets the class imbalance the paper attacks with a low decoding threshold |
| Decoding | `argmax`, plus a τ-threshold sweep for rare classes | **`argmax` only** — focal loss is the training-side counterpart of their threshold sweep |
| Architectures | UNet (from scratch), PSPNet, DeepLabv3 | **VGG-16 U-Net** (ImageNet-pretrained *and* from scratch) + our **`custom_lwir`** |

The **brightness-jitter exclusion** is shared and well-justified: in LWIR
the absolute pixel intensity *is* the class signal (a warm body is the only
cue for `living_obs`), so photometric augmentation actively destroys
information. The paper notes this in Section 5.1; we adopted it from the
start.

### Comparison against the paper's UNet baseline

The closest direct comparison is the paper's UNet (from scratch, native
resolution) versus our **`vgg16_scratch`** at 384×480. The paper reports
the following F1 scores at τ = 0.6 / τ = 0.3 (their Table 7):

| Class | UNet F1@0.6 | UNet F1@0.3 |
|---|---:|---:|
| sky | 98.0 | — |
| water | 100 | — |
| **bridge** | **58.8** | **71.3** |
| **obstacle** | **31.3** | **66.4** |
| **living_obs** | **16.1** | **54.5** |
| background | 92.7 | — |
| self | 94.8 | — |

The numbers tell the same story we will see in §6.4: the dominant classes
(`sky`, `water`, `background`, `self`) saturate near perfect F1 for *any*
reasonable architecture; the meaningful differences live in `bridge`,
`obstacle`, `living_obs`. The paper's UNet collapses on `living_obs` at the
standard τ = 0.6 (F1 = 16.1) because that 0.05 %-pixel class rarely wins an
`argmax` — they recover it by *lowering the decoding threshold*. We chose
the **training-time** counterpart instead — focal loss with γ = 2 — which
re-weights the loss toward exactly those hard, low-density pixels. The two
approaches target the same phenomenon from opposite ends of the pipeline,
and both are valid.

### Where `custom_lwir` sits in the landscape

The paper's architecture ranking (DeepLabv3 > PSPNet > UNet) reflects
*decoder sophistication*: DeepLabv3's atrous-spatial-pyramid-pooling head
and PSPNet's pyramid pooling provide stronger global context than UNet's
plain skip connections. Our `custom_lwir` brings that same idea to a
U-Net-style backbone via a **Transformer bottleneck**: global self-attention
at the deepest stride-16 feature map provides scene-wide context similar to
ASPP/PSP, but with ~5× fewer parameters than even a vanilla VGG-16 U-Net.
The result we expect to see in §6.6 is that `custom_lwir` ends up between
the paper's UNet and DeepLabv3 in per-class F1 on the rare classes — a
modernised U-Net that earns most of the global-context benefit at a small
fraction of the capacity.

### Where the augmentation effect should be largest

The paper notes (Section 6, "Larger rotations … caused distortion … smaller
range fit the dataset well"; "Shuffling was crucial"): the augmentation
effect on MassMIND is small in absolute terms but matters most for the
rare classes, which augmentation regularises. We expect the same pattern in
§6.5 — the augmentation delta should be larger for `vgg16_scratch` and
`custom_lwir` (which start without ImageNet priors) than for
`vgg16_pretrained`, and within each model the gains should concentrate on
`bridge` / `obstacle` rather than `sky` / `water` (which are already at
ceiling).

### Limitations & future work

* **Downsampling**: every absolute mIoU is lower than the paper's native-
  resolution numbers — small/thin structures (the parts of a bridge below
  the horizon, distant humans) lose detail. The *internal* comparisons
  (aug vs no-aug, model vs model) remain valid because all six runs share
  the resolution.
* **No threshold sweep**: we report only `argmax` per-class IoU/P/R. A
  threshold-based decoding (as the paper does) could lift `living_obs`
  recall further; we attacked the same problem from the loss side.
* **Sample size on `living_obs`**: this class is so rare (~0.05 % of
  pixels) that any single-run F1 estimate has high variance. The paper
  notes the same caveat.

In summary, the work delivers what the assignment asks for: a from-scratch
custom architecture (`custom_lwir`), a comparison against an existing model
(VGG-16 U-Net, in pretrained and from-scratch flavours), the required
metrics (IoU train + test, Precision, Recall, parameter count), an
augmentation ablation, and a results section grounded in the dataset's
published methodology.

### References

- Nirgudkar, S., DeFilippo, M., Sacarny, M., Benjamin, M., Robinette, P.
  (2023). *MassMIND: Massachusetts Maritime INfrared Dataset.* International
  Journal of Robotics Research, 42(1–2), 21–32.
  [doi:10.1177/02783649231153020](https://doi.org/10.1177/02783649231153020)
- Ronneberger, O., Fischer, P., Brox, T. (2015). *U-Net: Convolutional
  Networks for Biomedical Image Segmentation.* MICCAI.
- Lin, T.-Y., Goyal, P., Girshick, R., He, K., Dollár, P. (2017).
  *Focal Loss for Dense Object Detection.* ICCV.
- Howard, A. et al. (2017). *MobileNets: Efficient Convolutional Neural
  Networks for Mobile Vision Applications.* arXiv:1704.04861.
- Wu, Y., He, K. (2018). *Group Normalization.* ECCV.
- Vaswani, A. et al. (2017). *Attention Is All You Need.* NeurIPS.
- Chen, J. et al. (2021). *TransUNet: Transformers Make Strong Encoders
  for Medical Image Segmentation.* arXiv:2102.04306.
- Zhao, H. et al. (2017). *Pyramid Scene Parsing Network.* CVPR.
- Chen, L. et al. (2018). *DeepLab: Semantic Image Segmentation with Deep
  Convolutional Nets.* IEEE TPAMI.
